# 11 — Coherence 2D Pump Sweeps

Consolidates legacy experiments 21, 23, 24, 25.

1. **T1 vs. pump** — T1 relaxation measured over a 2D grid of pump power × detuning (Exp 21)
2. **T2 Ramsey vs. pump (wide detuning)** — Exp 23
3. **T2 Ramsey vs. pump (symmetric detuning)** — Exp 24
4. **T1 companion at same grid** — T1 at same pump settings as T2 (Exp 25)

Requires external SignalCore synthesizer for pump tone generation.

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    T1Relaxation,
    T2Ramsey,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="11_coherence_2d_pump_sweeps",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 06 status: {prev_checkpoint['status']}")

## 2. Pump Sweep Defaults

In [ ]:
# SignalCore synthesizer settings
# TODO: Update COM port and instrument abstraction for your setup
SIGNALCORE_COM_PORT = "COM3"  # Legacy hardcoded — update as needed

# Pump grid
PUMP_POWERS_DBM = np.arange(-20, 1, 2)         # dBm, 11 points
PUMP_DETUNINGS_MHZ = np.arange(-100, 101, 10)  # MHz, 21 points

# Symmetric detuning subset (for Exp 24)
PUMP_DETUNINGS_SYMMETRIC_MHZ = np.arange(-50, 51, 5)  # MHz, 21 points

# Per-point coherence measurement
T1_N_AVG = 5000
T2_N_AVG = 5000

print(f"Pump grid: {len(PUMP_POWERS_DBM)} powers × {len(PUMP_DETUNINGS_MHZ)} detunings = {len(PUMP_POWERS_DBM) * len(PUMP_DETUNINGS_MHZ)} points")
print(f"Symmetric grid: {len(PUMP_POWERS_DBM)} powers × {len(PUMP_DETUNINGS_SYMMETRIC_MHZ)} detunings")
print(f"T1 n_avg: {T1_N_AVG}, T2 n_avg: {T2_N_AVG}")

## 3. Configure SignalCore Synthesizer

Initialize the external pump source.

In [ ]:
# TODO: Replace with clean instrument abstraction when available
# For now, use the legacy COM-port based control

def set_pump(freq_hz, power_dbm):
    """Set the SignalCore pump tone frequency and power."""
    # Placeholder — wire up to actual instrument driver
    raise NotImplementedError(
        f"Wire up SignalCore on {SIGNALCORE_COM_PORT}: "
        f"freq={freq_hz/1e9:.6f} GHz, power={power_dbm} dBm"
    )

def pump_off():
    """Turn off the pump."""
    raise NotImplementedError("Wire up SignalCore pump-off")

# Qubit frequency for computing absolute pump frequency
f_qubit_hz = attr.qubit.f_01  # Hz

print(f"Qubit frequency: {f_qubit_hz / 1e9:.6f} GHz")
print("SignalCore control: placeholder — implement set_pump() and pump_off()")

## 4. T1 vs. Pump — Exp 21

2D sweep: at each (power, detuning), set the pump and measure T1.

In [ ]:
t1_vs_pump = np.full((len(PUMP_POWERS_DBM), len(PUMP_DETUNINGS_MHZ)), np.nan)

t1_exp = T1Relaxation(session)

for i, power_dbm in enumerate(PUMP_POWERS_DBM):
    for j, det_mhz in enumerate(PUMP_DETUNINGS_MHZ):
        pump_freq_hz = f_qubit_hz + det_mhz * 1e6
        set_pump(pump_freq_hz, power_dbm)

        result = t1_exp.run(n_avg=T1_N_AVG)
        analysis = t1_exp.analyze(result, update_calibration=False)
        t1_us = analysis.metrics.get("T1_us", np.nan)
        t1_vs_pump[i, j] = t1_us

    print(f"  Power {power_dbm:+.0f} dBm: row complete")

pump_off()
print("T1 vs. pump sweep complete.")

### T1 Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.pcolormesh(
    PUMP_DETUNINGS_MHZ, PUMP_POWERS_DBM, t1_vs_pump,
    shading="auto", cmap="viridis",
)
ax.set_xlabel("Pump detuning (MHz)")
ax.set_ylabel("Pump power (dBm)")
ax.set_title("T1 vs. Pump Tone")
fig.colorbar(im, ax=ax, label="T1 (μs)")
plt.tight_layout()
plt.show()

## 5. T2 Ramsey vs. Pump (Wide Detuning) — Exp 23

2D sweep: T2 Ramsey at each pump (power, detuning) — wide grid.

In [ ]:
t2_wide = np.full((len(PUMP_POWERS_DBM), len(PUMP_DETUNINGS_MHZ)), np.nan)

t2_exp = T2Ramsey(session)

for i, power_dbm in enumerate(PUMP_POWERS_DBM):
    for j, det_mhz in enumerate(PUMP_DETUNINGS_MHZ):
        pump_freq_hz = f_qubit_hz + det_mhz * 1e6
        set_pump(pump_freq_hz, power_dbm)

        result = t2_exp.run(n_avg=T2_N_AVG)
        analysis = t2_exp.analyze(result, update_calibration=False)
        t2_us = analysis.metrics.get("T2_us", np.nan)
        t2_wide[i, j] = t2_us

    print(f"  Power {power_dbm:+.0f} dBm: row complete")

pump_off()
print("T2 Ramsey vs. pump (wide) complete.")

## 6. T2 Ramsey vs. Pump (Symmetric Detuning) — Exp 24

Narrower, symmetric detuning grid for detailed AC Stark shift analysis.

In [ ]:
t2_symmetric = np.full((len(PUMP_POWERS_DBM), len(PUMP_DETUNINGS_SYMMETRIC_MHZ)), np.nan)

for i, power_dbm in enumerate(PUMP_POWERS_DBM):
    for j, det_mhz in enumerate(PUMP_DETUNINGS_SYMMETRIC_MHZ):
        pump_freq_hz = f_qubit_hz + det_mhz * 1e6
        set_pump(pump_freq_hz, power_dbm)

        result = t2_exp.run(n_avg=T2_N_AVG)
        analysis = t2_exp.analyze(result, update_calibration=False)
        t2_us = analysis.metrics.get("T2_us", np.nan)
        t2_symmetric[i, j] = t2_us

    print(f"  Power {power_dbm:+.0f} dBm: row complete")

pump_off()
print("T2 Ramsey vs. pump (symmetric) complete.")

## 7. T1 Companion at Same Grid — Exp 25

T1 at same pump settings as the symmetric T2 grid for direct comparison.

In [ ]:
t1_companion = np.full((len(PUMP_POWERS_DBM), len(PUMP_DETUNINGS_SYMMETRIC_MHZ)), np.nan)

for i, power_dbm in enumerate(PUMP_POWERS_DBM):
    for j, det_mhz in enumerate(PUMP_DETUNINGS_SYMMETRIC_MHZ):
        pump_freq_hz = f_qubit_hz + det_mhz * 1e6
        set_pump(pump_freq_hz, power_dbm)

        result = t1_exp.run(n_avg=T1_N_AVG)
        analysis = t1_exp.analyze(result, update_calibration=False)
        t1_us = analysis.metrics.get("T1_us", np.nan)
        t1_companion[i, j] = t1_us

    print(f"  Power {power_dbm:+.0f} dBm: row complete")

pump_off()
print("T1 companion sweep complete.")

## 8. AC Stark Shift Analysis

Compare T1 and T2 maps to identify AC Stark shift signatures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# T1 wide
im0 = axes[0].pcolormesh(
    PUMP_DETUNINGS_MHZ, PUMP_POWERS_DBM, t1_vs_pump,
    shading="auto", cmap="viridis",
)
axes[0].set_title("T1 vs. Pump (wide)")
axes[0].set_xlabel("Detuning (MHz)")
axes[0].set_ylabel("Power (dBm)")
fig.colorbar(im0, ax=axes[0], label="T1 (μs)")

# T2 wide
im1 = axes[1].pcolormesh(
    PUMP_DETUNINGS_MHZ, PUMP_POWERS_DBM, t2_wide,
    shading="auto", cmap="inferno",
)
axes[1].set_title("T2 Ramsey vs. Pump (wide)")
axes[1].set_xlabel("Detuning (MHz)")
axes[1].set_ylabel("Power (dBm)")
fig.colorbar(im1, ax=axes[1], label="T2 (μs)")

# T2 symmetric
im2 = axes[2].pcolormesh(
    PUMP_DETUNINGS_SYMMETRIC_MHZ, PUMP_POWERS_DBM, t2_symmetric,
    shading="auto", cmap="inferno",
)
axes[2].set_title("T2 Ramsey vs. Pump (symmetric)")
axes[2].set_xlabel("Detuning (MHz)")
axes[2].set_ylabel("Power (dBm)")
fig.colorbar(im2, ax=axes[2], label="T2 (μs)")

plt.tight_layout()
plt.show()

## 9. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="11_coherence_2d_pump_sweeps",
    status="characterized",
    summary="T1 and T2 Ramsey vs. external pump tone (2D power × detuning sweeps + AC Stark analysis).",
    consumed_inputs={
        "pump_powers_dbm": PUMP_POWERS_DBM.tolist(),
        "pump_detunings_mhz": PUMP_DETUNINGS_MHZ.tolist(),
        "pump_detunings_symmetric_mhz": PUMP_DETUNINGS_SYMMETRIC_MHZ.tolist(),
        "t1_n_avg": T1_N_AVG,
        "t2_n_avg": T2_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="12_chevron_experiments",
    notes=[
        "Requires external SignalCore synthesizer — set_pump() must be implemented.",
        "All measurements are characterization-only — no calibrations applied.",
        "AC Stark shift analysis is qualitative (visual comparison).",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")